This workzone dashboard is a work in progress.  It's goal is to present a user-friendly, attractive, and interactive way to display multiple exploration options for the data available.

In [124]:
import base64
import datashader
import geopandas as gpd
import holoviews as hv
import hvplot.pandas
import json
import numpy as np
import os
import pandas as pd
import panel as pn
import plotly.express as px
import plotly.graph_objects as go
import sqlite3
from colorcet import bmy
from PIL import Image

# Enable Panel extensions
pn.extension('plotly','vizzu', 'tabulator', design='material', template='fast')

The first section of the dashboard is the Introduction.  It provides a logo and title for the dashboard and gives introductory message regarding the contents of the page.

In [125]:
# Intro
instruction_text = """
<div style='color: black;'>
    <h3 style='font-weight: bold;'>Data Analysis and Visualization of Work Zone Collisions</h3>
    This dashboard visualizes collision locations within construction work zones along Kentucky highways and explores some
    of the statistics by comparing them to each other and to roadway characteristics by looking at variables such as age
    and gender of persons involved, number and categories of vehicles, weather, and road conditions.
</div>
"""

instruction = pn.pane.Markdown(instruction_text, width=600)

# Create the workzone_logo with the correct pane method
workzone_logo = pn.pane.Image('Report_Images/KYTCWorkZoneSafety.png', width=200, align='center')

# Add custom CSS
pn.config.raw_css = [""".white-background {background-color: white;}"""]

# Create the row with white background
intro = pn.Row(workzone_logo, instruction, sizing_mode='stretch_width', css_classes=['white-background'])

The next pane to be added is the dataframe.  The data displayed is based on a SQL query joining the ksp_incidents to the Roadway_Characteristics_API table data.  Not all columns of either table are included here, but can easily be expanded by changing the SQL query to include other parameters.

In [126]:
# Define the path for the SQLite database
database_path = 'data/crash_data.db'

# Check if the database file exists
if not os.path.exists(database_path):
    print(f"Error: The database file '{database_path}' does not exist.")
else:
    try:
        # Connect to the database
        conn = sqlite3.connect(database_path)

        query = """
                SELECT i.IncidentID, i.County,strftime('%Y', i.CollisionDate) AS Year,
                       i.CollisionDate AS Date, strftime('%t', i.CollisionDate) AS Time,
                       p.AgeAtIncident AS Driver_Age, p.Gender AS Driver_Gender,
                       i.MotorVehiclesInvolved as Num_Vehicles,
                       i.NumberKilled AS Fatalities, i.NumberInjured AS Injuries,
                       i.Weather, i.RdwyConditionCode AS Rdwy_Condition,
                       i.MannerofCollision, i.RdwyCharacter, i.LightCondition,
                       r.Road_Name, r.Milepoint, r.Speed_Limit_Posted_MPH AS Speed_Limit
                FROM ksp_incidents AS i
                JOIN Roadway_Characteristics_API AS r
                    ON i.IncidentID = r.IncidentID
                JOIN ksp_person AS p
                    ON i.IncidentID = p.IncidentID
                WHERE r.Route_Type IN ('I', 'PKWY', 'US', 'KY')
                    AND p.PersonTypeCde = 1
                """

        # Execute the query and fetch the results into a DataFrame
        df = pd.read_sql_query(query, conn)

        # Read in the Roadway Characteristics data from geojson file
        rdwy_df = pd.read_csv('data/api_clean_data/RoadwayCharacteristics.csv')

        # Merge the DataFrame with the GeoDataFrame on 'IncidentID'
        merged_df = df.merge(rdwy_df[['IncidentID', 'Latitude', 'Longitude']], on='IncidentID', how='left')

        # Remove rows where 'Latitude' or 'Longitude' is NaN or null
        cleaned_df = merged_df[merged_df['Latitude'].notna() & merged_df['Longitude'].notna()]

        print(cleaned_df.head())

    except sqlite3.OperationalError as e:
        print(f"OperationalError: {e}")


   IncidentID     County  Year        Date  Time  Driver_Age Driver_Gender  \
0    31442377    MADISON  2023  2023-03-31  None         NaN          None   
1    32273654    BULLITT  2023  2023-09-30  None         NaN          None   
2    32392281  JEFFERSON  2023  2023-10-26  None         NaN          None   
3    33179195    GRAYSON  2023  2023-08-03  None        43.0        FEMALE   
4    33302773      CASEY  2023  2023-09-07  None        25.0        FEMALE   

   Num_Vehicles  Fatalities  Injuries  Weather Rdwy_Condition  \
0             2           0         0  RAINING            WET   
1             2           0         0   CLOUDY            DRY   
2             2           0         0    CLEAR            DRY   
3             2           0         0    CLEAR            DRY   
4             2           0         0    CLEAR            DRY   

          MannerofCollision     RdwyCharacter        LightCondition  \
0  SIDESWIPE-SAME DIRECTION  STRAIGHT & LEVEL                  DAWN  

This cell stores the incidents returned as a cache to provide imporved performance.

In [127]:
pn.state.cache.clear()  # Clears the cache

incidents = pn.state.as_cached('incidents', lambda: cleaned_df)

incidents.head()


,IncidentID,County,Year,Date,Time,Driver_Age,Driver_Gender,Num_Vehicles,Fatalities,Injuries,Weather,Rdwy_Condition,MannerofCollision,RdwyCharacter,LightCondition,Road_Name,Milepoint,Speed_Limit,Latitude,Longitude
0,31442377,MADISON,2023,2023-03-31,None,NaN,None,2,0,0,RAINING,WET,SIDESWIPE-SAME DIRECTION,STRAIGHT & LEVEL,DAWN,I-75 NC,78.079,70.0,37.605387,-84.314743
1,32273654,BULLITT,2023,2023-09-30,None,NaN,None,2,0,0,CLOUDY,DRY,REAR END,STRAIGHT & LEVEL,DAYLIGHT,I-65,118.612,70.0,38.020144,-85.696360
2,32392281,JEFFERSON,2023,2023-10-26,None,NaN,None,2,0,0,CLEAR,DRY,SIDESWIPE-SAME DIRECTION,STRAIGHT & LEVEL,DAYLIGHT,I-265,28.457,65.0,38.265728,-85.501310
3,33179195,GRAYSON,2023,2023-08-03,None,43.0,FEMALE,2,0,0,CLEAR,DRY,SIDESWIPE-SAME DIRECTION,STRAIGHT & LEVEL,DARK-HWY NOT LIGHTED,WENDELL H FORD-WESTERN KENTUCKY PKWY NC,114.291,70.0,37.499459,-86.167144
4,33302773,CASEY,2023,2023-09-07,None,25.0,FEMALE,2,0,0,CLEAR,DRY,REAR END,CURVE & GRADE,DAYLIGHT,KY-70,5.544,55.0,37.284540,-85.083500


The next pane creates the dataframe pane for the visualization.

In [128]:
# Create custom CSS for smaller font size and applies the CSS to the Tabulator widget
custom_css = """.tabulator .tabulator-cell {font-size: 12px;}"""
pn.config.raw_css.append(custom_css)

# Build the Panel pane layout
df_pane = pn.widgets.Tabulator(df, sizing_mode='stretch_width', height=300)


The following cell trys to emulate the Wind Turbines dashboard on the Panels tutorial site for plotting data.  It isn't perfect, but it is a start to understand the workings of binding different components together in an interactive way.

In [131]:
ls = hv.link_selections.instance()

geo = ls(incidents.hvplot.points(
    'Longitude', 'Latitude', xaxis=None, yaxis=None, rasterize=True,
    tiles='CartoLight', responsive=True, dynspread=True,
    height=500, cnorm='log', cmap='plasma',
    xlim=(-89.0, -81.0),  # Longitude bounds for Kentucky
    ylim=(36.5, 39.0)      # Latitude bounds for Kentucky
))

# Display the plot
geo



BokehModel(combine_events=True, render_bundle={'docs_json': {'615e555a-47c3-4bff-82bc-7feb6026592d': {'version…

Task exception was never retrieved
future: <Task finished name='Task-34641' coro=<Callback.on_change() done, defined at c:\Users\Teri.Dowdy\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone32\lib\site-packages\holoviews\plotting\bokeh\callbacks.py:318> exception=UnsetValueError("figure(id='24abee4a-7754-43e3-aaa4-ad8eccb61288', ...).inner_height doesn't have a value set")>
Traceback (most recent call last):
  File "c:\Users\Teri.Dowdy\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone32\lib\site-packages\holoviews\plotting\bokeh\callbacks.py", line 385, in process_on_change
    msg[attr] = self.resolve_attr_spec(path, cb_obj)
  File "c:\Users\Teri.Dowdy\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone32\lib\site-packages\holoviews\plotting\bokeh\callbacks.py", line 298, in resolve_attr_spec
    resolved = getattr(resolved, p, None)
  File "C:\Users\Teri.Dowdy\AppData\Roaming\Python\Python39\site-packages\bokeh\core\property\descriptors.py", line 283, in __get__
    raise UnsetValueErro

In [ ]:
def data(df, groupby, quant):
    if quant == 'Count':
        return df.value_counts(groupby).to_frame(name='Count').sort_index().reset_index().iloc[:50]
    else:
        return df.groupby(groupby)[quant].sum().reset_index().iloc[:50]

def config(chart_type, groupby, quant):
    if chart_type == 'Pie Chart':
        return {
            "channels": {
                "x": None,
                "y": None,
                "color": groupby,
                "label": groupby,
                "size": quant
            },
            'geometry': 'circle'
        }
    else:
        return {
            "channels": {
                "x": groupby,
                "y": quant,
                "color": None,
                "label": None,
                "size": None
            },
            'geometry': 'rectangle'
        }



groupby = pn.widgets.RadioButtonGroup(
    options={'County': 'county', 'Year': 'year', 'Gender': 'driver_gender'}, align='center'
)
chart_type = pn.widgets.RadioButtonGroup(
    options=['Bar Chart', 'Pie Chart'], align='center'
)
quant = pn.widgets.RadioButtonGroup(
    options={'Count': 'Count', 'Capacity': 'p_cap'}, align='center'
)
lsdata = ls.selection_param(incidents)

vizzu = pn.pane.Vizzu(
    pn.bind(data, lsdata, groupby, quant),
    config=pn.bind(config, chart_type, groupby, quant),
    column_types={'year': 'dimension'},
    style={
        "plot": {
            "xAxis": {
                "label": {
                    "angle": "-45deg"
                }
            }
        }
    },
    sizing_mode='stretch_both'
)

def format_df(df):
    df = df[['county', 'year', 'driver_age', 'driver_gender', 'injuries', 'fatalities']]
    return df.rename(
        columns={col: col.split('_')[1].title() for col in df.columns}
    )


table = pn.widgets.Tabulator(
    pn.bind(format_df, lsdata), page_size=8, pagination='remote',
    show_index=False,
)

pn.Column(
    pn.Row(quant, "# by", groupby, "# as a", chart_type).servable(area='header'),
    pn.Column(
        pn.Row(geo, table),
        vizzu, min_height=1000,
        sizing_mode='stretch_both'
    ).servable(title='Incidents')
)

This last cell builds the Panel layout by combining the intro, df_pane, and the title. The layout is then displayed in the default browser window.

In [ ]:
# Create the final app layout
app = pn.Column(pn.layout.Divider(), "### Introduction", intro,
                pn.layout.Divider(), "### Incident Data", df_pane
                # pn.layout.Divider(), "### Location", map_pane,
)

# Serve the app
app.servable()
app.show()